## Tool definition

In [2]:
## Esto no lo corri lo adapte a ollama local 
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

# Verificación de keys SIN imprimir su contenido (repo público)
k = os.getenv("LANGSMITH_API_KEY") or ""
print("LANGSMITH key presente:", bool(k), "| largo esperado 51:", len(k) == 51)
print("TAVILY key presente:", bool(os.getenv("TAVILY_API_KEY")))
print("ENDPOINT:", os.getenv("LANGSMITH_ENDPOINT"))


In [2]:
from langchain.tools import tool

@tool #decorador de la tool
def square_root(x: float) -> float: #funcion que se usara
    """Calculate the square root of a number""" # que es la tool 
    return x ** 0.5

In [3]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [4]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [5]:
tool1.invoke({"x": 467})

21.61018278497431

In [6]:
square_root.invoke({"x": 467})

21.61018278497431

In [11]:
from local_model import get_model, OLLAMA_URL
print(OLLAMA_URL)

http://<ollama-host>:11434


In [12]:
import requests, time
t = time.time()
print(requests.get(OLLAMA_URL, timeout=10).text)          # debe decir "Ollama is running"
r = requests.post(f"{OLLAMA_URL}/api/chat", timeout=180, json={
    "model": "qwen3:8b", "stream": False, "think": False,
    "messages": [{"role": "user", "content": "Say hi"}]})
print(round(time.time() - t, 1), "s →", r.json()["message"]["content"])

Ollama is running
20.9 s → Hello! How can I assist you today? 😊


In [13]:
print(requests.get(f"{OLLAMA_URL}/api/ps", timeout=10).json())
print([m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags", timeout=10).json()["models"]])

{'models': [{'name': 'qwen3:8b', 'model': 'qwen3:8b', 'size': 5937267997, 'digest': '500a1f067a9f782620b40bee6f7b0c89e17ae61f686b92c24933e4ca4b2b8b41', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen3', 'families': ['qwen3'], 'parameter_size': '8.2B', 'quantization_level': 'Q4_K_M'}, 'expires_at': '2026-10-03T17:00:50.6527002-05:00', 'size_vram': 0, 'context_length': 4096}]}
['gemma4:latest', 'qwen3:8b', 'qwen3:4b', 'qwen3:30b-a3b', 'gemma3:4b', 'tinyllama:latest', 'Llama2:latest', 'gemma3:latest']


## Adding to agents

In [ ]:
from local_model import get_model, OLLAMA_URL
#model = get_model()                      # qwen3:8b en <ollama-host>

In [14]:
from langchain.agents import create_agent
from local_model import get_model, OLLAMA_URL # lo adicione para que funcione con ollama

agent = create_agent(
    model=get_model(), # lo adicione para que funcione con ollama
    tools=[tool1],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [15]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The square root of 467 is approximately **21.61**. 

If you'd like the square of 467 as well, it is **217,489**. Let me know if you need further calculations!


In [16]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='What is the square root of 467?', additional_kwargs={}, response_metadata={}, id='997b3ecd-075c-41c3-aca1-3d47814d4723'),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-10-03T22:01:33.4318002Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13683813400, 'load_duration': 5078570400, 'prompt_eval_count': 168, 'prompt_eval_duration': 4919049000, 'eval_count': 22, 'eval_duration': 3671024000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--01a103c9-3230-7352-a251-0f56a40cc202-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': 'd21e7d46-f097-4fda-8c11-57e8a55e1744', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 168, 'output_tokens': 22, 'total_tokens': 190}),
 ToolMessage(content='21.61018278497431', name='square_root', id='13c9d9eb-ebfd-40c4-a864-890eede35ee2', tool_call_id='d21e7d46-f097-4fda-8c11-57e8a55e1744'),
 AIM

In [17]:
print(response["messages"][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 467}, 'id': 'd21e7d46-f097-4fda-8c11-57e8a55e1744', 'type': 'tool_call'}]
